In [18]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate
from torchtext.vocab import build_vocab_from_iterator

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'：将 Hugging Face 的 API 端点设置为国内镜像站。官方源 huggingface.co 服务器在海外，国内访问经常超时或无法连接。设置镜像后，datasets 库的所有下载请求会被重定向到镜像站，解决网络问题，提升下载速度和稳定性。
dataset = datasets.load_dataset("bentrevett/multi30k")：使用 datasets 库从 Hugging Face Hub 加载 Multi30k 数据集。这是一个包含约3万条英德平行句对的翻译数据集。函数会自动完成下载、缓存和解析，返回包含训练集、验证集和测试集的 DatasetDict 对象。

In [6]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [8]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [9]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [10]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [11]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


In [12]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


In [13]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [14]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [19]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)
de_vocab = build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [20]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [21]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [22]:
en_vocab["the"]

7

In [23]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [24]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [25]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [26]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

"crime" 被转换成 <unk> 的原因是：

词频过滤（min_freq = 2）：在构建词汇表时，设置了 min_freq=2，这意味着只有在训练集中出现至少2次的单词才会被加入词汇表。

"crime" 出现次数不足：单词 "crime" 在 Multi30k 训练集中出现的次数少于2次（可能只出现了1次或0次）。Multi30k 是一个图像描述数据集，主要包含日常场景的描述，像 "crime" 这样的词汇可能确实很少出现。

未知词处理：当词汇表遇到不在其中的单词时，会使用默认索引（通过 set_default_index(unk_index) 设置），将其映射为 <unk> 标记。

转换过程：

lookup_indices(["crime"]) 返回 [0]（unk_index）
lookup_tokens([0]) 返回 ["<unk>"]

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


In [27]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [28]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

第一个单元格（Cell 27）- numericalize_example 函数：

这个函数的作用是将文本 token 转换为数字索引（数值化）：

接收一个样本 example，包含英语和德语的 token 列表
使用 en_vocab.lookup_indices() 将英语 token 列表转换为索引列表
使用 de_vocab.lookup_indices() 将德语 token 列表转换为索引列表
返回包含 en_ids 和 de_ids 的字典，这些 ID 是神经网络可以处理的数字形式
第二个单元格（Cell 28）- 应用数值化：

这个单元格将 numericalize_example 函数应用到所有数据集：

使用 .map() 方法批量处理训练集、验证集和测试集
通过 fn_kwargs 传递词汇表参数
将所有样本的 token 列表转换为数字索引列表
转换后，每个样本会新增 en_ids 和 de_ids 字段，包含可以直接输入模型的数字序列
总结： 这两个单元格完成了从文本到数字的关键转换步骤。例如：

Token: ['<sos>', 'i', 'love', 'cats', '<eos>']
IDs: [2, 956, 2169, 1234, 3]

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [29]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [31]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [32]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

In [33]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [34]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [35]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

第一个函数：get_collate_fn(pad_index)

这是一个高阶函数（返回另一个函数的函数），用于创建批处理整理函数：

外层函数 get_collate_fn：接收 pad_index（填充索引）作为参数
内层函数 collate_fn：实际的批处理整理函数，作用是：
从一个批次的多个样本中提取所有英语 ID 序列（batch_en_ids）
从一个批次的多个样本中提取所有德语 ID 序列（batch_de_ids）
使用 nn.utils.rnn.pad_sequence 对序列进行填充对齐：因为每个句子长度不同，需要用 <pad> 标记填充到相同长度，使它们能组成一个张量
返回包含填充后的英语和德语 ID 张量的字典
第二个函数：get_data_loader(dataset, batch_size, pad_index, shuffle=False)

这个函数创建 PyTorch 的 DataLoader 对象：

调用 get_collate_fn(pad_index) 获取整理函数
创建 torch.utils.data.DataLoader，配置：
dataset：要加载的数据集
batch_size：每批次的样本数量
collate_fn：使用上面创建的整理函数来处理批次数据
shuffle：是否打乱数据顺序（训练时通常为 True，验证/测试时为 False）
返回配置好的 DataLoader，可以迭代获取批次数据
总结： 这两个函数协同工作，将数据集转换为可迭代的批次加载器。关键功能是自动填充不同长度的序列，使它们能够批量处理。例如：

句子1：[2, 16, 24, 3] (长度4)
句子2：[2, 100, 200, 300, 400, 3] (长度6)
填充后变成：

句子1：[2, 16, 24, 3, 1, 1] (长度6，用pad_index=1填充)
句子2：[2, 100, 200, 300, 400, 3] (长度6)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

In [37]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

让我详细解释这个 Encoder 类的代码：

Encoder 类详解
1. 输入参数（__init__ 方法）
input_dim：输入词汇表的大小（例如：英语词汇表有 5000 个单词）
embedding_dim：词嵌入向量的维度（例如：256，将每个单词映射为 256 维向量）
hidden_dim：LSTM 隐藏状态的维度（例如：512）
n_layers：LSTM 的层数（例如：2 层堆叠的 LSTM）
dropout：Dropout 比例（例如：0.5，用于防止过拟合）
2. 核心组件
① 词嵌入层（Embedding Layer）

self.embedding = nn.Embedding(input_dim, embedding_dim)
将单词索引转换为稠密的向量表示
例如：单词索引 100 → 256 维向量 [0.2, -0.5, 0.8, ...]
② LSTM 层（Long Short-Term Memory）

self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
处理序列数据，捕捉上下文信息
多层堆叠（n_layers）增强表达能力
层间使用 Dropout 防止过拟合
③ Dropout 层

self.dropout = nn.Dropout(dropout)
应用于嵌入层输出，随机丢弃部分神经元
提高模型泛化能力
3. Forward 函数处理流程
输入：

src：源语言句子的索引序列，形状 [src_length, batch_size]
例如：[[2, 2, 2], [16, 100, 50], [24, 200, 60], [3, 3, 3]] 表示 3 个句子，每个长度为 4
处理步骤：

步骤 1：词嵌入 + Dropout

embedded = self.dropout(self.embedding(src))
# embedded = [src_length, batch_size, embedding_dim]
将每个单词索引转换为嵌入向量
应用 Dropout 正则化
步骤 2：LSTM 处理

outputs, (hidden, cell) = self.rnn(embedded)
LSTM 逐时间步处理嵌入序列
返回三个张量：
outputs：每个时间步的输出（来自最顶层 LSTM）
hidden：所有层的最终隐藏状态
cell：所有层的最终细胞状态
4. 输出
return hidden, cell
返回值：

hidden：形状 [n_layers, batch_size, hidden_dim]

包含所有 LSTM 层的最终隐藏状态
编码了整个源句子的语义信息
cell：形状 [n_layers, batch_size, hidden_dim]

包含所有 LSTM 层的最终细胞状态
LSTM 的内部记忆状态
注意： outputs 没有返回，因为在这个 Seq2Seq 架构中，只需要最终的 hidden 和 cell 状态传递给 Decoder。

5. 工作原理示例
假设输入句子："I love cats"

索引化：[2, 956, 2169, 1234, 3]（包含 <sos> 和 <eos>）
嵌入：每个索引 → 256 维向量
LSTM 处理：逐词读取，更新隐藏状态
最终输出：hidden 和 cell 包含整个句子的压缩表示，传递给 Decoder 用于生成翻译
这个 Encoder 的作用是将可变长度的源语言句子编码成固定大小的上下文向量（hidden 和 cell），供 Decoder 解码生成目标语言。

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

Decoder 工作流程详解
1. 初始化参数
output_dim：输出词汇表大小（目标语言词汇表，如德语词汇表有 7000 个单词）
embedding_dim：词嵌入维度（与 Encoder 相同，如 256）
hidden_dim：LSTM 隐藏状态维度（必须与 Encoder 相同，如 512）
n_layers：LSTM 层数（必须与 Encoder 相同，如 2 层）
dropout：Dropout 比例
2. 核心组件
词嵌入层：nn.Embedding(output_dim, embedding_dim) - 将目标语言单词索引转换为向量
LSTM 层：nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout) - 生成序列
全连接输出层：nn.Linear(hidden_dim, output_dim) - 将 LSTM 输出映射到词汇表大小，用于预测下一个单词
Dropout 层：防止过拟合
3. Forward 函数工作流程
输入：

input：当前时间步的单词索引，形状 [batch_size]（一次只处理一个单词）
hidden：上一时间步的隐藏状态，形状 [n_layers, batch_size, hidden_dim]
cell：上一时间步的细胞状态，形状 [n_layers, batch_size, hidden_dim]
处理步骤：

步骤 1：增加序列维度

input = input.unsqueeze(0)
 input = [1, batch_size]
将 [batch_size] 扩展为 [1, batch_size]，因为 LSTM 需要序列长度维度
步骤 2：词嵌入 + Dropout

embedded = self.dropout(self.embedding(input))
 embedded = [1, batch_size, embedding_dim]
将单词索引转换为嵌入向量
应用 Dropout 正则化
步骤 3：LSTM 处理

output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
 output = [1, batch_size, hidden_dim]
 hidden = [n_layers, batch_size, hidden_dim]
 cell = [n_layers, batch_size, hidden_dim]
将嵌入向量和上一时间步的状态输入 LSTM
生成当前时间步的输出和更新后的状态
步骤 4：预测下一个单词

prediction = self.fc_out(output.squeeze(0))
 prediction = [batch_size, output_dim]
移除序列维度：squeeze(0) 将 [1, batch_size, hidden_dim] 变为 [batch_size, hidden_dim]
通过全连接层映射到词汇表大小，得到每个单词的概率分数
输出：

prediction：形状 [batch_size, output_dim]，包含词汇表中每个单词的得分
hidden：更新后的隐藏状态，传递给下一时间步
cell：更新后的细胞状态，传递给下一时间步

In [38]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

In [45]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

Seq2Seq 类详解
1. 初始化（__init__ 方法）
def __init__(self, encoder, decoder, device):
参数：

encoder：编码器实例
decoder：解码器实例
device：运行设备（CPU 或 GPU）
关键检查：

assert encoder.hidden_dim == decoder.hidden_dim
assert encoder.n_layers == decoder.n_layers
确保 Encoder 和 Decoder 的隐藏层维度和层数必须相同
这是因为 Encoder 的最终状态要直接传递给 Decoder 作为初始状态
2. Forward 函数流程
输入：

src：源语言句子，形状 [src_length, batch_size]
trg：目标语言句子，形状 [trg_length, batch_size]
teacher_forcing_ratio：Teacher Forcing 使用概率（如 0.75 表示 75% 的时间使用真实标签）
处理流程：

步骤 1：初始化输出张量

outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
创建一个全零张量来存储每个时间步的预测结果
步骤 2：编码源句子

hidden, cell = self.encoder(src)
将源语言句子输入 Encoder
获得编码后的上下文向量（hidden 和 cell）
这些状态包含了整个源句子的语义信息
步骤 3：初始化 Decoder 输入

input = trg[0, :]  # 取目标句子的第一个 token，即 <sos>
Decoder 的第一个输入是 <sos>（句子开始标记）
步骤 4：逐步生成目标句子

for t in range(1, trg_length):
    output, hidden, cell = self.decoder(input, hidden, cell)
    outputs[t] = output

    teacher_force = random.random() < teacher_forcing_ratio
    top1 = output.argmax(1)
    input = trg[t] if teacher_force else top1
循环中的每一步：

解码：将当前输入和状态传入 Decoder，得到预测和更新后的状态
存储预测：将预测结果保存到 outputs[t]
Teacher Forcing 决策：随机决定是否使用 Teacher Forcing
获取最佳预测：从预测中选择概率最高的单词（argmax）
选择下一个输入：
如果使用 Teacher Forcing：使用真实的下一个单词 trg[t]
如果不使用：使用模型预测的单词 top1
步骤 5：返回所有预测

return outputs  # [trg_length, batch_size, trg_vocab_size]
3. Teacher Forcing 机制
什么是 Teacher Forcing？

Teacher Forcing 是一种训练技巧，在训练时使用真实的目标序列作为 Decoder 的输入，而不是使用模型自己的预测。

工作原理：

teacher_force = random.random() < teacher_forcing_ratio
input = trg[t] if teacher_force else top1
示例：翻译 "I love cats" → "Ich liebe Katzen"

假设 teacher_forcing_ratio = 0.75：

时间步	当前输入	模型预测	Teacher Force?	下一个输入
1
<sos>
"Ich"
是 (75%)
"Ich" (真实)
2
"Ich"
"liebe"
是 (75%)
"liebe" (真实)
3
"liebe"
"Hunde" (错误)
否 (25%)
"Hunde" (预测)
4
"Hunde"
"sind"
是 (75%)
<eos> (真实)
优点：

✅ 加速训练：使用真实标签可以更快地收敛
✅ 稳定训练：避免早期错误预测累积导致训练崩溃
缺点：

⚠️ 训练-推理不一致：训练时看到真实标签，推理时只能用自己的预测
⚠️ 过度依赖：模型可能过度依赖真实输入，无法处理自己的错误
解决方案：

使用 0.5-0.75 的 teacher_forcing_ratio，平衡两种模式
训练后期逐渐降低该比例，让模型适应自己的预测
4. 完整工作流程总结
Encoder 读取源句子 → 生成上下文向量（hidden, cell）
Decoder 接收上下文向量和 <sos> → 开始生成
循环生成：每步预测一个单词，根据 Teacher Forcing 决定下一个输入
返回结果：所有时间步的预测概率分布
这个 Seq2Seq 模型将 Encoder 和 Decoder 整合在一起，实现了完整的序列到序列翻译功能。

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [46]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)
# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)
# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [47]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [48]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [49]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [50]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [52]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 将模型设置为训练模式（启用 Dropout 等）
    model.train()

    # 初始化累计损失
    epoch_loss = 0

    # 遍历数据加载器中的每个批次
    for i, batch in enumerate(data_loader):
        # 获取源语言（德语）数据并移动到设备（CPU/GPU）
        src = batch["de_ids"].to(device)

        # 获取目标语言（英语）数据并移动到设备
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]

        # 清空梯度，防止累积
        optimizer.zero_grad()

        # 前向传播：将源语言和目标语言输入模型，得到预测输出
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]

        # 获取输出的最后一个维度大小（词汇表大小）
        output_dim = output.shape[-1]

        # 重塑输出：去掉第一个时间步（<sos>），展平为二维张量
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]

        # 重塑目标：去掉第一个时间步（<sos>），展平为一维张量
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]

        # 计算损失：比较模型预测和真实标签
        loss = criterion(output, trg)

        # 反向传播：计算梯度
        loss.backward()

        # 梯度裁剪：防止梯度爆炸，将梯度限制在 [-clip, clip] 范围内
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # 更新模型参数
        optimizer.step()

        # 累加当前批次的损失
        epoch_loss += loss.item()

    # 返回平均损失（总损失 / 批次数量）
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [53]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

# 模型训练

In [54]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|██████████| 1/1 [06:54<00:00, 414.70s/it]

	Train Loss:   5.049 | Train PPL: 155.831
	Valid Loss:   5.021 | Valid PPL: 151.517


# 模型验证

In [55]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [56]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [57]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [58]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [59]:
translation

['<sos>', 'a', 'man', 'in', 'a', 'a', 'and', 'a', 'a', '.', '<eos>']